In [1]:
# ── CELL 1: Install dependencies ──────────────────────────
!pip install -q transformers accelerate peft bitsandbytes torch
!pip install -q fastapi uvicorn pyngrok nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.2 MB/s eta 0:00:00


In [2]:
# ── CELL 2: Mount Drive & load model ──────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import json

from google.colab import drive
drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/question-generation-mistral"
BASE_MODEL  = "mistralai/Mistral-7B-v0.1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA weights...")
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

print("✅ Model ready!")

Mounted at /content/drive
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loading LoRA weights...
✅ Model ready!


In [3]:
# ── CELL 3: Helper functions ───────────────────────────────
def parse_generated_question(generated_text):
    """Parse raw model output into structured dict."""
    try:
        if "### Question:" not in generated_text:
            return None
        parts = generated_text.split("### Question:")[1]

        if "### Choices:" not in parts:
            return None
        question_text  = parts.split("### Choices:")[0].strip()
        choices_section = parts.split("### Choices:")[1]

        if "### Correct Answer:" not in choices_section:
            return None
        choices_text   = choices_section.split("### Correct Answer:")[0].strip()
        correct_letter = choices_section.split("### Correct Answer:")[1].strip()[0]

        choices = []
        for line in choices_text.split('\n'):
            if line.strip() and ')' in line:
                choice = line.split(')', 1)[1].strip()
                choices.append(choice)

        correct_idx = ord(correct_letter.upper()) - ord('A')

        if not question_text or len(choices) < 2:
            return None

        return {
            "question": question_text,
            "choices":  choices,
            "correct":  correct_idx,
        }
    except Exception as e:
        print(f"Parse error: {e}")
        return None


def generate_questions(transcript: str, num_questions: int = 5, temperature: float = 0.8):
    # Split transcript into chunks of ~2800 chars with slight overlap
    chunk_size = 2800
    overlap    = 200
    chunks = []
    start  = 0
    while start < len(transcript):
        end = start + chunk_size
        chunks.append(transcript[start:end])
        start = end - overlap

    # Distribute questions across chunks evenly
    # e.g. 5 questions, 3 chunks → chunks get [2, 2, 1] questions
    questions_per_chunk = [num_questions // len(chunks)] * len(chunks)
    for i in range(num_questions % len(chunks)):
        questions_per_chunk[i] += 1

    print(f"  Transcript split into {len(chunks)} chunks.")
    print(f"  Questions per chunk: {questions_per_chunk}")

    all_questions = []
    q_counter = 0

    for chunk_idx, (chunk, n_q) in enumerate(zip(chunks, questions_per_chunk)):
        for i in range(n_q):
            q_counter += 1
            parsed = None
            max_retries = 2

            for attempt in range(max_retries):
                prompt = f"""### Passage:
{chunk.strip()}

### Task:
Generate question #{q_counter} - a multiple-choice question based on the passage above.

### Question:"""

                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                ).to("cuda")

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=250,
                        temperature=temperature + (attempt * 0.1),
                        do_sample=True,
                        top_p=0.95,
                        pad_token_id=tokenizer.eos_token_id,
                    )

                generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                parsed = parse_generated_question(generated_text)

                if parsed:
                    break
                else:
                    print(f"  ⚠️ Q{q_counter} chunk {chunk_idx+1} attempt {attempt+1} failed, retrying...")

            if parsed:
                all_questions.append(parsed)
            else:
                print(f"  ❌ Q{q_counter} failed after {max_retries} attempts, skipping.")

    return all_questions

In [4]:
# ── CELL 3b: Debug helper + robust parser ─────────────────
# Replace parse_generated_question in Cell 3 with this version,
# or just run this cell AFTER Cell 3 to override it.

import re

def parse_generated_question(generated_text):
    """
    Robust parser — handles slight variations in model output formatting.
    Logs the raw text if parsing fails so you can see what went wrong.
    """
    try:
        # ── Strip the prompt (everything up to the last ### Question:) ──
        if "### Question:" not in generated_text:
            print(f"  [DEBUG] No '### Question:' found in output:\n{generated_text[:300]}\n")
            return None

        # Take only the part after the LAST "### Question:" (strips the prompt)
        parts = generated_text.rsplit("### Question:", 1)[1]

        # ── Extract question text ────────────────────────────────────────
        # Try strict header first, then fallback to first newline block
        if "### Choices:" in parts:
            question_text = parts.split("### Choices:")[0].strip()
            rest = parts.split("### Choices:")[1]
        else:
            # Fallback: question is the first non-empty line
            lines = [l.strip() for l in parts.strip().split('\n') if l.strip()]
            if not lines:
                print(f"  [DEBUG] Empty output after ### Question:\n{parts[:300]}\n")
                return None
            question_text = lines[0]
            rest = '\n'.join(lines[1:])

        # ── Extract choices ──────────────────────────────────────────────
        if "### Correct Answer:" in rest:
            choices_block  = rest.split("### Correct Answer:")[0]
            correct_block  = rest.split("### Correct Answer:")[1]
        else:
            choices_block  = rest
            correct_block  = ""

        # Parse lines like "A) ...", "A. ...", "A: ..." or "1) ..."
        choices = []
        for line in choices_block.split('\n'):
            line = line.strip()
            if not line:
                continue
            # Match: A) / A. / A: / 1) / 1. at the start
            m = re.match(r'^[A-Da-d1-4][)\.\:]\s*(.+)', line)
            if m:
                choices.append(m.group(1).strip())

        if len(choices) < 2:
            print(f"  [DEBUG] Not enough choices parsed. choices_block was:\n{choices_block[:300]}\n")
            return None

        # ── Extract correct answer ───────────────────────────────────────
        correct_idx = 0  # default to A if we can't find it
        if correct_block.strip():
            letter = correct_block.strip()[0].upper()
            if letter in "ABCD":
                correct_idx = ord(letter) - ord('A')
            elif letter in "1234":
                correct_idx = int(letter) - 1

        # Clamp to valid range
        correct_idx = min(correct_idx, len(choices) - 1)

        if not question_text:
            print(f"  [DEBUG] Empty question text.\n")
            return None

        return {
            "question": question_text,
            "choices":  choices,
            "correct":  correct_idx,
        }

    except Exception as e:
        print(f"  [DEBUG] Parser exception: {e}")
        print(f"  [DEBUG] Raw text was:\n{generated_text[:400]}\n")
        return None

In [5]:
# ── CELL 3c: Load Evaluation Model ────────────────────────
# Run this AFTER Cell 3b (question generation model is already loaded)

EVAL_MODEL_PATH = "/content/drive/MyDrive/adaptive-learning-evaluation-FINAL"

print("Loading evaluation model tokenizer...")
eval_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
eval_tokenizer.pad_token = eval_tokenizer.eos_token
eval_tokenizer.padding_side = "right"

print("Loading evaluation base model (4-bit)...")
eval_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

eval_base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=eval_bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading evaluation LoRA weights...")
eval_model = PeftModel.from_pretrained(eval_base_model, EVAL_MODEL_PATH)
eval_model.eval()

print("✅ Evaluation model ready!")

Loading evaluation model tokenizer...
Loading evaluation base model (4-bit)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading evaluation LoRA weights...
✅ Evaluation model ready!


In [6]:
# ── CELL 3d: Evaluation helper functions ──────────────────

def build_eval_prompt(transcript: str, questions: list, student_answers: list, preference: str) -> str:
    """
    Build the prompt the evaluation model was trained on.
    """
    # Truncate transcript to fit context window
    context = transcript.strip()[:2800]

    qa_section = ""
    for i, (q, student_ans) in enumerate(zip(questions, student_answers), 1):
        qa_section += f"Q{i}: {q['question']}\nStudent Answer: {student_ans}\n\n"

    return f"""### Domain: [GENERAL]
### Context:
{context}

### Assessment:
{qa_section}### User Preference: {preference}

### Evaluation:"""


def parse_eval_output(generated_text: str):
    """
    Parse evaluation model output into evaluation + recommendation.
    Returns dict with 'evaluation' and 'recommendation' keys.
    """
    try:
        if "### Evaluation:" not in generated_text:
            return None

        after_eval = generated_text.split("### Evaluation:")[1].strip()

        if "### Recommendation:" in after_eval:
            evaluation   = after_eval.split("### Recommendation:")[0].strip()
            recommendation = after_eval.split("### Recommendation:")[1].strip()
        else:
            # Model didn't generate recommendation section — use full output as evaluation
            evaluation     = after_eval.strip()
            recommendation = ""

        # Clean up any trailing prompt artifacts
        for marker in ["### Domain:", "### Context:", "### Assessment:"]:
            if marker in evaluation:
                evaluation = evaluation.split(marker)[0].strip()
            if marker in recommendation:
                recommendation = recommendation.split(marker)[0].strip()

        return {
            "evaluation":     evaluation,
            "recommendation": recommendation,
        }

    except Exception as e:
        print(f"  [DEBUG] Eval parse error: {e}")
        print(f"  [DEBUG] Raw text: {generated_text[:400]}")
        return None


def run_evaluation(transcript: str, questions: list, student_answers: list, preference: str):
    """
    Run the evaluation model and return structured result.
    """
    prompt = build_eval_prompt(transcript, questions, student_answers, preference)

    inputs = eval_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to("cuda")

    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            do_sample=True,
            top_p=0.95,
            pad_token_id=eval_tokenizer.eos_token_id,
        )

    generated_text = eval_tokenizer.decode(outputs[0], skip_special_tokens=True)
    parsed = parse_eval_output(generated_text)

    if parsed is None:
        # Fallback so the API never returns empty
        parsed = {
            "evaluation":     generated_text.split("### Evaluation:")[-1][:300].strip(),
            "recommendation": "",
        }

    return parsed

print("✅ Evaluation helper functions ready!")


✅ Evaluation helper functions ready!


In [8]:
# ── CELL 4: FastAPI app ────────────────────────────────────
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import uvicorn
import nest_asyncio
import requests
from pyngrok import ngrok

class EvaluateRequest(BaseModel):
    transcript:      str
    questions:       list          # list of {"question", "choices", "correct"}
    student_answers: list          # list of answer strings matching questions order
    preference:      Optional[str] = "videos"  # "videos" or "articles"


class EvaluateResponse(BaseModel):
    evaluation:     str
    recommendation: str
    score:          int
    total:          int
    passed:         bool

nest_asyncio.apply()

app = FastAPI(title="Question Generation API")


class GenerateRequest(BaseModel):
    transcript:    str
    num_questions: Optional[int] = 5
    temperature:   Optional[float] = 0.8


class QuestionResponse(BaseModel):
    questions: list
    count:     int


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_PATH}


@app.post("/generate", response_model=QuestionResponse)
def generate(req: GenerateRequest):
    if not req.transcript.strip():
        raise HTTPException(status_code=400, detail="Transcript is empty.")

    if req.num_questions < 1 or req.num_questions > 20:
        raise HTTPException(status_code=400, detail="num_questions must be between 1 and 20.")

    print(f"\n→ Generating {req.num_questions} questions (temp={req.temperature})...")
    questions = generate_questions(req.transcript, req.num_questions, req.temperature)
    print(f"✅ Returned {len(questions)} questions.")

    return {"questions": questions, "count": len(questions)}

@app.post("/evaluate", response_model=EvaluateResponse)
def evaluate(req: EvaluateRequest):
    if not req.transcript.strip():
        raise HTTPException(status_code=400, detail="Transcript is empty.")

    if len(req.questions) != len(req.student_answers):
        raise HTTPException(status_code=400, detail="Questions and answers length mismatch.")

    if req.preference not in ("videos", "articles"):
        raise HTTPException(status_code=400, detail="preference must be 'videos' or 'articles'.")

    # Calculate score
    score = 0
    for q, student_ans in zip(req.questions, req.student_answers):
        correct_ans = q["choices"][q["correct"]]
        if student_ans == correct_ans:
            score += 1

    total  = len(req.questions)
    passed = score >= (total * 0.6)  # 60% threshold to advance

    print(f"\n→ Evaluating: {score}/{total} correct, preference={req.preference}...")

    result = run_evaluation(
        req.transcript,
        req.questions,
        req.student_answers,
        req.preference,
    )

    print(f"✅ Evaluation done.")

    return {
        "evaluation":     result["evaluation"],
        "recommendation": result["recommendation"],
        "score":          score,
        "total":          total,
        "passed":         passed,
        }


In [9]:
# ── CELL 5: Start server + ngrok tunnel ───────────────
# Paste your ngrok auth token here (from https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = "3AIhGm2Ih8LkOrw7B44HG8NKU5S_7C516SgaGBYDDWxdALaaB"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open tunnel on port 8000
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"  ✅ API is live at: {public_url}")
print(f"  Health check:     {public_url}/health")
print(f"  Generate endpoint:{public_url}/generate")
print("="*60)
print("\nCopy the URL above into your Streamlit app's API_URL variable.\n")

# Start server (blocks — keep this cell running)
import threading
import time
import nest_asyncio

def run_uvicorn_server():
    # Apply nest_asyncio again in this thread to handle uvicorn's event loop
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

print("\nStarting FastAPI server in a background thread...")
uvicorn_thread = threading.Thread(target=run_uvicorn_server, daemon=True)
uvicorn_thread.start()

print("Server started. This cell will now block to keep the server alive.")
print("To stop the server, interrupt this cell's execution.")

try:
    # Keep the main thread alive so the daemon server thread continues to run
    while True:
        time.sleep(3600) # Sleep for a long time (1 hour)
except KeyboardInterrupt:
    print("\nServer stopped by user via KeyboardInterrupt.")
    pass

INFO:     Started server process [1588]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



  ✅ API is live at: NgrokTunnel: "https://dulotic-fumigatory-romona.ngrok-free.dev" -> "http://localhost:8000"
  Health check:     NgrokTunnel: "https://dulotic-fumigatory-romona.ngrok-free.dev" -> "http://localhost:8000"/health
  Generate endpoint:NgrokTunnel: "https://dulotic-fumigatory-romona.ngrok-free.dev" -> "http://localhost:8000"/generate

Copy the URL above into your Streamlit app's API_URL variable.


Starting FastAPI server in a background thread...
Server started. This cell will now block to keep the server alive.
To stop the server, interrupt this cell's execution.

→ Generating 5 questions (temp=0.8)...
  Transcript split into 7 chunks.
  Questions per chunk: [1, 1, 1, 1, 1, 0, 0]
✅ Returned 5 questions.
INFO:     46.193.67.14:0 - "POST /generate HTTP/1.1" 200 OK

→ Evaluating: 4/5 correct, preference=videos...
✅ Evaluation done.
INFO:     46.193.67.14:0 - "POST /evaluate HTTP/1.1" 200 OK

Server stopped by user via KeyboardInterrupt.
